# Avance 4 — AUTH Recommendation Validity vs Current AUTH History REAL v4

**Objetivo:** validar si las recomendaciones vigentes del baseline siguen siendo defendibles contra la historia AUTH actual real, sin correr el modelo.

Esta versión agrega una lógica más madura para evitar falsos positivos:

- Distingue **bandas válidas** vs **bandas colapsadas** (`P20 = P85`) o de bajo soporte.
- Compara **posición baseline vs posición current** para separar extremos persistentes de extremos nuevos.
- Agrega el estado **Watch**, para no tratar como Yellow accionable todo lo que solo requiere vigilancia.
- Separa estados de monitoreo: calidad de datos, cobertura de recomendación, cobertura de bins/catálogo, drift de historia AUTH y vigencia de recomendación.
- Incluye perfiles de sensibilidad: `conservative`, `balanced`, `strict`.

**Nota de alcance:** los catálogos de bins fueron creados con datos completos (`AUTH + no AUTH`). En este notebook se evalúa cómo los registros/snapshots AUTH caen dentro de ese catálogo global versionado.


**v4 update:** agrega una capa de decisión operativa para separar `scoring/update`, `HB-SVI retraining`, `catalog review` y `keep/watch`.

**Pipeline abstraction copy:** este notebook es la copia de transicion para extraer la logica a componentes. El notebook original queda como referencia del analista; esta version debe ir reemplazando constantes y exportacion directa por contratos compartidos con el pipeline.


In [ ]:
# =========================================================
# 0. Imports and notebook configuration
# =========================================================

import os
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pricing_mlops.io import materialize_monitoring_outputs, write_artifact_manifest

from pricing.auth_monitoring import (
    calculate_auth_history_drift,
    calculate_operational_decision,
    calculate_recommendation_validity,
    expected_auth_monitoring_artifacts,
    load_auth_monitoring_config,
    validate_expected_monitoring_artifacts,
)

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
# Shared monitoring contract used by the Azure ML pipeline components.
PIPELINE_MONITORING_CONFIG = load_auth_monitoring_config()
MONITORING_CONFIG = PIPELINE_MONITORING_CONFIG
MONITORING_ARTIFACTS = expected_auth_monitoring_artifacts()


## 1. Configuración de la corrida

Esta sección define el alcance del monitoreo, las rutas de entrada/salida y las rutas de entrada/salida y umbrales de monitoreo. 

Puntos importantes:

- `MONITORING_SCOPE = AUTH_ONLY`: el análisis se interpreta como historia de autorizaciones.
- `BIN_CATALOG_SCOPE = AUTH_AND_NON_AUTH_GLOBAL`: los bins vienen de catálogos construidos con todos los datos.
- `RUNS_MODEL = False`: este notebook no ejecuta el modelo ni genera nuevas recomendaciones.


In [ ]:
# =========================================================
# 1. Run configuration
# =========================================================

# ---------------------------------------------------------
# Override / Experimentos
# ---------------------------------------------------------
# La config oficial del pipeline se conserva en PIPELINE_MONITORING_CONFIG.
# Cambia solo estos valores para probar sensibilidad en el notebook; deja None
# para usar el valor oficial.
# Guia completa: docs/auth-monitoring-configuration.md
#
# Ejemplo de experimento local:
#   EXPERIMENTAL_THRESHOLDS["psi_yellow"] = 0.15
#   EXPERIMENTAL_THRESHOLDS["new_combo_rate_red"] = 0.20
# Eso solo afecta esta ejecución del notebook. El pipeline seguirá usando los
# valores oficiales mientras no se modifique auth_monitoring_config.json.
EXPERIMENTAL_THRESHOLDS = {
    "psi_yellow": None,
    "psi_red": None,
    "ks_yellow": None,
    "ks_red": None,
    "current_history_coverage_yellow": None,
    "current_history_coverage_red": None,
    "new_combo_rate_yellow": None,
    "new_combo_rate_red": None,
    "global_watch_rate_threshold": None,
    "global_watch_revenue_share_threshold": None,
    "scoring_update_new_combo_rate_threshold": None,
    "scoring_update_new_combo_count_threshold": None,
}

MONITORING_CONFIG = PIPELINE_MONITORING_CONFIG.with_overrides(thresholds=EXPERIMENTAL_THRESHOLDS)
ACTIVE_EXPERIMENTAL_THRESHOLDS = {
    key: value for key, value in EXPERIMENTAL_THRESHOLDS.items() if value is not None
}

PROJECT_NAME = MONITORING_CONFIG.project.name
TEAM = MONITORING_CONFIG.project.team
ENVIRONMENT = MONITORING_CONFIG.project.environment

MONITORING_SCOPE = MONITORING_CONFIG.project.monitoring_scope
CURRENT_HISTORY_INTERPRETATION = MONITORING_CONFIG.project.current_history_interpretation
BIN_CATALOG_SCOPE = MONITORING_CONFIG.project.bin_catalog_scope
BIN_CATALOG_VERSION = MONITORING_CONFIG.project.bin_catalog_version
RUNS_MODEL = MONITORING_CONFIG.project.runs_model

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc)
DRIFT_RUN_ID = RUN_TIMESTAMP_UTC.strftime("%Y%m%dT%H%M%SZ") + "_auth_recommendation_validity_v4"

OUTPUT_ROOT = Path("auth_monitoring_outputs") / DRIFT_RUN_ID
OUTPUT_DIRS = {
    "snapshots": OUTPUT_ROOT / "snapshots",
    "logs": OUTPUT_ROOT / "logs",
    "summaries": OUTPUT_ROOT / "summaries",
    "reports": OUTPUT_ROOT / "reports",
    "figures": OUTPUT_ROOT / "figures",
    "manifest": OUTPUT_ROOT / "manifest",
}
for path in OUTPUT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

BASELINE_SNAPSHOT_PATH = Path(
    "masked_data_inputcomplete_inputauth_output/avance3_outputs/20260530T194311Z_baseline_v1/snapshots/model_output_snapshot.csv"
)
CURRENT_AUTH_HISTORY_PATH = Path(
    "auth_history_preparation_outputs/20260609T203037Z_new_history_prepare_clean_v1/history/current_auth_history_snapshot_real.csv"
)
SEARCH_ROOTS = [
    Path("auth_history_preparation_outputs"),
    Path("/mnt/data/auth_history_preparation_outputs"),
    Path("/mnt/data"),
]

MIN_SEGMENT_SUPPORT_FOR_DASHBOARD = MONITORING_CONFIG.thresholds.min_segment_support_for_dashboard
ALERT_PROFILE = MONITORING_CONFIG.thresholds.alert_profile

NEAR_BAND_EDGE_THRESHOLD = MONITORING_CONFIG.thresholds.near_band_edge_threshold
GAP_VS_P50_YELLOW = MONITORING_CONFIG.thresholds.gap_vs_p50_yellow
GAP_VS_P50_RED = MONITORING_CONFIG.thresholds.gap_vs_p50_red
P50_SHIFT_YELLOW = MONITORING_CONFIG.thresholds.p50_shift_yellow
P50_SHIFT_RED = MONITORING_CONFIG.thresholds.p50_shift_red
COLLAPSED_GAP_YELLOW = MONITORING_CONFIG.thresholds.collapsed_gap_yellow
COLLAPSED_GAP_RED = MONITORING_CONFIG.thresholds.collapsed_gap_red

ROW_COUNT_RATIO_WARNING_LOW = 0.80
ROW_COUNT_RATIO_WARNING_HIGH = 1.20
CURRENT_HISTORY_COVERAGE_RED_THRESHOLD = MONITORING_CONFIG.thresholds.current_history_coverage.red
CURRENT_HISTORY_COVERAGE_YELLOW_THRESHOLD = MONITORING_CONFIG.thresholds.current_history_coverage.yellow
NEW_COMBO_RATE_YELLOW_THRESHOLD = MONITORING_CONFIG.thresholds.new_combo_rate.yellow
NEW_COMBO_RATE_RED_THRESHOLD = MONITORING_CONFIG.thresholds.new_combo_rate.red
PSI_YELLOW = MONITORING_CONFIG.thresholds.psi.yellow
PSI_RED = MONITORING_CONFIG.thresholds.psi.red
KS_YELLOW = MONITORING_CONFIG.thresholds.ks.yellow
KS_RED = MONITORING_CONFIG.thresholds.ks.red

CATALOG_BIN_MISSING_RATE_PASS = 0.10
CATALOG_BIN_MISSING_RATE_WARNING = 0.25
GLOBAL_WATCH_RATE_THRESHOLD = MONITORING_CONFIG.thresholds.global_watch_rate_threshold
GLOBAL_WATCH_REVENUE_SHARE_THRESHOLD = MONITORING_CONFIG.thresholds.global_watch_revenue_share_threshold
SCORING_UPDATE_NEW_COMBO_RATE_THRESHOLD = MONITORING_CONFIG.thresholds.scoring_update_new_combo_rate_threshold
SCORING_UPDATE_NEW_COMBO_COUNT_THRESHOLD = MONITORING_CONFIG.thresholds.scoring_update_new_combo_count_threshold

print("Drift run ID:", DRIFT_RUN_ID)
print("Output root:", OUTPUT_ROOT)
print("Monitoring scope:", MONITORING_SCOPE)
print("Runs model:", RUNS_MODEL)
print("Alert profile:", ALERT_PROFILE)
print("Experimental threshold overrides:", ACTIVE_EXPERIMENTAL_THRESHOLDS or "None")


## 2. Carga del baseline oficial de Avance 3

El input principal es el `model_output_snapshot.csv` de Avance 3. Este archivo contiene la recomendación vigente, percentiles históricos AUTH y señales operativas generadas por la corrida baseline.

Si el notebook no encuentra automáticamente el archivo, asigna manualmente la variable `BASELINE_SNAPSHOT_PATH` en la celda anterior.


In [ ]:
# =========================================================
# 2. Load baseline model_output_snapshot from Avance 3
# =========================================================

def find_latest_snapshot(search_roots):
    candidates = []
    for root in search_roots:
        if root.exists():
            candidates.extend(root.rglob("model_output_snapshot.csv"))
    candidates = [p for p in candidates if "avance3_outputs" in str(p).lower() or "snapshots" in str(p).lower()]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)

if BASELINE_SNAPSHOT_PATH is None:
    BASELINE_SNAPSHOT_PATH = find_latest_snapshot(SEARCH_ROOTS)

if BASELINE_SNAPSHOT_PATH is None or not Path(BASELINE_SNAPSHOT_PATH).exists():
    raise FileNotFoundError(
        "Could not find model_output_snapshot.csv. "
        "Set BASELINE_SNAPSHOT_PATH manually in section 1."
    )

BASELINE_SNAPSHOT_PATH = Path(BASELINE_SNAPSHOT_PATH)
baseline_snapshot = pd.read_csv(BASELINE_SNAPSHOT_PATH)

print("Baseline snapshot path:", BASELINE_SNAPSHOT_PATH)
print("Baseline shape:", baseline_snapshot.shape)
display(baseline_snapshot.head())


## 3. Definición de llaves, recomendación vigente e historia AUTH baseline

En este proyecto la granularidad principal de monitoreo es:

`kpn + vpareadescription + distysegment`

La recomendación oficial puede venir de varias columnas. El orden de prioridad usado es:

1. `Selected_Optimal_Price`, si existe.
2. `selected_recommended_price`, si existe.
3. `Balanced`, como proxy.
4. `More_Profit`, como fallback.
5. `Revenue_Aggressive`, como fallback final.

Para esta etapa, si se usa `Balanced`, debe entenderse como proxy de la recomendación vigente. En producción debe usarse la estrategia oficial aprobada por Pricing/PM.


In [ ]:
# =========================================================
# 3. Build baseline recommendation snapshot and baseline AUTH history profile
# =========================================================

KEY_COLUMNS = list(MONITORING_CONFIG.columns.key_columns)
missing_keys = [c for c in KEY_COLUMNS if c not in baseline_snapshot.columns]
if missing_keys:
    raise ValueError(f"Missing key columns in baseline snapshot: {missing_keys}")

RECOMMENDATION_CANDIDATES = list(dict.fromkeys(
    list(MONITORING_CONFIG.columns.official_recommendation_preferred)
    + list(MONITORING_CONFIG.columns.proxy_recommendation)
))

RECOMMENDATION_PRICE_COLUMN = next((c for c in RECOMMENDATION_CANDIDATES if c in baseline_snapshot.columns), None)
if RECOMMENDATION_PRICE_COLUMN is None:
    raise ValueError(
        "No recommendation price column found. Expected one of: "
        + ", ".join(RECOMMENDATION_CANDIDATES)
    )

print("Recommendation price column used:", RECOMMENDATION_PRICE_COLUMN)

# Shared contract columns plus notebook inspection columns expected from baseline snapshot.
BASELINE_HISTORY_COLUMNS = list(dict.fromkeys(list(MONITORING_CONFIG.columns.baseline_history) + [
    "P20_PRICE", "P50_PRICE", "P85_PRICE", "P100_PRICE", "P0_PRICE",
    "P20_RESALE", "P50_RESALE", "P85_RESALE",
    "P20_QTY", "P50_QTY", "P85_QTY",
    "P20", "P50", "P85",
    "Min_P20_for_5pct_margin", "P20_Adjusted_Min5pctMargin", "P20_Was_Adjusted",
    "q0", "q0_local_obs", "n_transactions", "n_invoices",
    "revenue_sum", "quantity_sum", "revenue_median",
    "resale_price_median", "resale_price_mean", "into_stock_price_median",
    "log_quantity_mean", "log_into_stock_price_mean",
    "distributor_margin_pct_median", "channel_margin_share_median", "kemet_margin_pct_median",
    "negative_distributor_margin_rate", "negative_kemet_variable_margin_rate",
    "channel_margin_share_out_of_range_rate", "auth_cost_match_rate",
    "date_min", "date_max", "n_distributor_parents",
    "distributor_parentnumber_mode", "custombusinessgroup_mode",
    "price_band_width", "price_band_is_collapsed", "price_band_is_valid",
]))

# Bin columns: shared contract columns plus notebook inspection columns.
GLOBAL_BIN_COLUMNS = list(dict.fromkeys(list(MONITORING_CONFIG.columns.global_bin) + [
    "q0_local_obs_bin",
    "order_size_bin_mode",
    "disty_margin_original_bin_mode",
    "channel_margin_share_bin_mode",
    "revenue_weight_bucket",
    "recommended_historical_zone",
    "recommendation_status_final",
    "review_priority_final",
]))

history_cols_available = [c for c in BASELINE_HISTORY_COLUMNS if c in baseline_snapshot.columns]
bin_cols_available = [c for c in GLOBAL_BIN_COLUMNS if c in baseline_snapshot.columns]

baseline_recommendation_snapshot = baseline_snapshot[
    KEY_COLUMNS + [RECOMMENDATION_PRICE_COLUMN]
    + [c for c in ["run_id", "baseline_version", "output_schema_version", "run_timestamp_utc"] if c in baseline_snapshot.columns]
].copy()

baseline_recommendation_snapshot = baseline_recommendation_snapshot.rename(
    columns={RECOMMENDATION_PRICE_COLUMN: "baseline_recommended_price"}
)

baseline_auth_history_profile = baseline_snapshot[
    KEY_COLUMNS + history_cols_available + bin_cols_available
].copy()

# Add explicit interpretation fields.
baseline_recommendation_snapshot["recommendation_source_column"] = RECOMMENDATION_PRICE_COLUMN
baseline_recommendation_snapshot["monitoring_scope"] = MONITORING_SCOPE
baseline_auth_history_profile["history_scope"] = MONITORING_SCOPE
baseline_auth_history_profile["bin_catalog_scope"] = BIN_CATALOG_SCOPE
baseline_auth_history_profile["bin_catalog_version"] = BIN_CATALOG_VERSION

print("Baseline recommendation snapshot shape:", baseline_recommendation_snapshot.shape)
print("Baseline AUTH history profile shape:", baseline_auth_history_profile.shape)
print("Available global bin columns:", bin_cols_available)

display(baseline_recommendation_snapshot.head())
display(baseline_auth_history_profile.head())


### 3.1 Interpretación de la columna de recomendación

El monitoreo necesita una única columna que represente la recomendación vigente. En producción, esta debería ser la recomendación oficialmente seleccionada por Pricing/PM, por ejemplo `Selected_Optimal_Price`.

Si esa columna no existe, el notebook usa `Balanced` como proxy metodológico. Esto es aceptable para el MVP académico, pero debe quedar registrado para trazabilidad porque el semáforo puede cambiar si negocio selecciona otra estrategia como recomendación oficial.


In [ ]:
# =========================================================
# 3.1 Recommendation source interpretation
# =========================================================

official_recommendation_columns_available = [
    col for col in OFFICIAL_RECOMMENDATION_PREFERRED_COLUMNS
    if col in baseline_snapshot.columns
]

is_official_recommendation_source = RECOMMENDATION_PRICE_COLUMN in official_recommendation_columns_available
is_proxy_recommendation_source = not is_official_recommendation_source

recommendation_source_metadata = {
    "recommendation_source_column": RECOMMENDATION_PRICE_COLUMN,
    "is_official_recommendation_source": bool(is_official_recommendation_source),
    "is_proxy_recommendation_source": bool(is_proxy_recommendation_source),
    "source_interpretation": (
        "Official selected recommendation column was used."
        if is_official_recommendation_source
        else "Proxy recommendation column was used. Replace with official PM-selected recommendation in production."
    ),
    "monitoring_scope": MONITORING_SCOPE,
    "runs_model": RUNS_MODEL,
}

recommendation_source_metadata_df = pd.DataFrame([recommendation_source_metadata])
display(recommendation_source_metadata_df)


## 4. Carga de historia AUTH actual real

Esta sección reemplaza la simulación. Ahora el notebook carga el archivo `current_auth_history_snapshot_real.csv` generado por el notebook limpio de preparación de datos nuevos.

Este archivo ya debe venir:

- enmascarado con el mapping anterior extendido;
- con bins aplicados desde el catálogo global creado con AUTH + no AUTH;
- filtrado y agregado al scope AUTH;
- a nivel `kpn + vpareadescription + distysegment`;
- sin elasticidades ni recomendaciones nuevas, porque no se corrió el modelo.

El objetivo sigue siendo pre-model / pre-scoring:

> comparar la recomendación vigente del baseline contra la historia AUTH actual real.


In [ ]:
# =========================================================
# 4. Load real current AUTH history without running the model
# =========================================================

def find_latest_current_auth_history(search_roots):
    """
    Finds the most recently modified current_auth_history_snapshot_real.csv
    generated by the new history preparation notebook.
    """
    candidates = []
    for root in search_roots:
        if root.exists():
            for candidate in root.rglob("current_auth_history_snapshot_real.csv"):
                # Avoid accidentally loading outputs generated by this monitoring notebook.
                # The real current snapshot should come from the preparation notebook folder.
                if "auth_monitoring_outputs" in candidate.parts:
                    continue
                candidates.append(candidate)
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)

if CURRENT_AUTH_HISTORY_PATH is None:
    CURRENT_AUTH_HISTORY_PATH = find_latest_current_auth_history(SEARCH_ROOTS)

if CURRENT_AUTH_HISTORY_PATH is None or not Path(CURRENT_AUTH_HISTORY_PATH).exists():
    raise FileNotFoundError(
        "Could not find current_auth_history_snapshot_real.csv. "
        "Set CURRENT_AUTH_HISTORY_PATH manually in section 1 using the output from the preparation notebook."
    )

CURRENT_AUTH_HISTORY_PATH = Path(CURRENT_AUTH_HISTORY_PATH)

# Safety check: this file should come from the preparation notebook, not from a previous Avance 4 run.
if "auth_monitoring_outputs" in CURRENT_AUTH_HISTORY_PATH.parts:
    raise ValueError(
        "CURRENT_AUTH_HISTORY_PATH points to an Avance 4 monitoring output folder. "
        "Set it to the file generated by the preparation notebook under "
        "auth_history_preparation_outputs/.../snapshots/current_auth_history_snapshot_real.csv"
    )

current_auth_history = pd.read_csv(CURRENT_AUTH_HISTORY_PATH)

# Ensure key columns have consistent type for joins.
for col in KEY_COLUMNS:
    if col in current_auth_history.columns:
        current_auth_history[col] = current_auth_history[col].astype(str)
    if col in baseline_recommendation_snapshot.columns:
        baseline_recommendation_snapshot[col] = baseline_recommendation_snapshot[col].astype(str)
    if col in baseline_auth_history_profile.columns:
        baseline_auth_history_profile[col] = baseline_auth_history_profile[col].astype(str)

# Add/standardize metadata fields.
current_auth_history["history_scope"] = MONITORING_SCOPE
current_auth_history["history_snapshot_type"] = "current_auth_history_real"
current_auth_history["runs_model"] = RUNS_MODEL
current_auth_history["bin_catalog_scope"] = BIN_CATALOG_SCOPE
current_auth_history["bin_catalog_version"] = BIN_CATALOG_VERSION
current_auth_history["source_snapshot_path"] = str(CURRENT_AUTH_HISTORY_PATH)

# Keep readable date columns in CSV and compatible with drift checks.
for date_col in ["date_min", "date_max"]:
    if date_col in current_auth_history.columns:
        current_auth_history[date_col] = pd.to_datetime(current_auth_history[date_col], errors="coerce").astype(str)

# Summary replacing the old simulation summary.
baseline_keys = baseline_auth_history_profile[KEY_COLUMNS].drop_duplicates()
current_keys = current_auth_history[KEY_COLUMNS].drop_duplicates()

matched_current_keys = current_keys.merge(
    baseline_keys.assign(_in_baseline=1),
    on=KEY_COLUMNS,
    how="left"
)
new_combo_count = int(matched_current_keys["_in_baseline"].isna().sum())
new_combo_rate = float(new_combo_count / max(len(current_keys), 1))

matched_baseline_keys = baseline_keys.merge(
    current_keys.assign(_in_current=1),
    on=KEY_COLUMNS,
    how="left"
)
current_history_coverage_rate_preview = float(matched_baseline_keys["_in_current"].notna().mean())

current_history_summary = {
    "baseline_rows": int(len(baseline_auth_history_profile)),
    "current_rows": int(len(current_auth_history)),
    "baseline_unique_keys": int(len(baseline_keys)),
    "current_unique_keys": int(len(current_keys)),
    "new_combos": new_combo_count,
    "new_combo_rate": new_combo_rate,
    "current_history_coverage_rate_preview": current_history_coverage_rate_preview,
    "runs_model": RUNS_MODEL,
    "monitoring_scope": MONITORING_SCOPE,
    "history_snapshot_type": "current_auth_history_real",
    "current_auth_history_path": str(CURRENT_AUTH_HISTORY_PATH),
    "bin_catalog_scope": BIN_CATALOG_SCOPE,
    "bin_catalog_version": BIN_CATALOG_VERSION,
}

print(json.dumps(current_history_summary, indent=2))
print("Current AUTH history path:", CURRENT_AUTH_HISTORY_PATH)
print("Current AUTH history shape:", current_auth_history.shape)
display(current_auth_history.head())


## 5. Data Quality, Recommendation Coverage and Catalog Coverage Logs

Esta sección separa tres conceptos que antes podían confundirse:

- **Data quality:** estructura mínima del archivo, llaves, granularidad y percentiles.
- **Recommendation coverage:** cobertura de recomendaciones baseline contra historia actual y combos nuevos sin recomendación.
- **Catalog/bin coverage:** porcentaje de valores Missing al aplicar/arrastrar bins globales sobre el scope AUTH.

Los combos nuevos no se tratan como mala calidad; se tratan como una señal de cobertura que puede justificar una corrida de scoring/recommendation.


In [ ]:
# =========================================================
# 5. Data quality, recommendation coverage and catalog coverage logs
# =========================================================

quality_records = []
coverage_records = []
catalog_coverage_records = []

def add_quality_check(dataset_name, check_name, check_status, observed_value=None, threshold_value=None, details=None):
    quality_records.append({
        "drift_run_id": DRIFT_RUN_ID,
        "dataset_name": dataset_name,
        "check_name": check_name,
        "check_status": check_status,
        "observed_value": observed_value,
        "threshold_value": threshold_value,
        "details": details,
        "timestamp_utc": RUN_TIMESTAMP_UTC.isoformat(),
        "monitoring_scope": MONITORING_SCOPE,
    })


def add_coverage_check(check_name, check_status, observed_value=None, threshold_value=None, details=None):
    coverage_records.append({
        "drift_run_id": DRIFT_RUN_ID,
        "monitoring_stage": "recommendation_coverage_pre_model",
        "check_name": check_name,
        "check_status": check_status,
        "observed_value": observed_value,
        "threshold_value": threshold_value,
        "details": details,
        "timestamp_utc": RUN_TIMESTAMP_UTC.isoformat(),
        "monitoring_scope": MONITORING_SCOPE,
    })


def add_catalog_coverage_check(bin_column, missing_rate, details=None):
    if pd.isna(missing_rate):
        status = "Not_Evaluable"
    elif missing_rate > CATALOG_BIN_MISSING_RATE_WARNING:
        status = "Red"
    elif missing_rate > CATALOG_BIN_MISSING_RATE_PASS:
        status = "Yellow"
    else:
        status = "Green"
    catalog_coverage_records.append({
        "drift_run_id": DRIFT_RUN_ID,
        "monitoring_stage": "catalog_bin_coverage_pre_model",
        "bin_column": bin_column,
        "missing_rate": missing_rate,
        "coverage_rate": None if pd.isna(missing_rate) else 1 - missing_rate,
        "coverage_status": status,
        "threshold_pass_missing_rate": CATALOG_BIN_MISSING_RATE_PASS,
        "threshold_warning_missing_rate": CATALOG_BIN_MISSING_RATE_WARNING,
        "details": details,
        "timestamp_utc": RUN_TIMESTAMP_UTC.isoformat(),
        "monitoring_scope": MONITORING_SCOPE,
        "bin_catalog_scope": BIN_CATALOG_SCOPE,
        "bin_catalog_version": BIN_CATALOG_VERSION,
    })

# Required columns.
required_baseline_cols = KEY_COLUMNS + ["baseline_recommended_price"]
missing_baseline_rec_cols = [c for c in required_baseline_cols if c not in baseline_recommendation_snapshot.columns]
add_quality_check(
    "baseline_recommendation_snapshot",
    "required_recommendation_columns_present",
    "PASS" if not missing_baseline_rec_cols else "FAIL",
    observed_value=len(missing_baseline_rec_cols),
    threshold_value=0,
    details=f"Missing columns: {missing_baseline_rec_cols}",
)

required_history_cols = KEY_COLUMNS + ["P20_PRICE", "P50_PRICE", "P85_PRICE"]
missing_current_history_cols = [c for c in required_history_cols if c not in current_auth_history.columns]
add_quality_check(
    "current_auth_history",
    "required_current_history_columns_present",
    "PASS" if not missing_current_history_cols else "FAIL",
    observed_value=len(missing_current_history_cols),
    threshold_value=0,
    details=f"Missing columns: {missing_current_history_cols}",
)

# Row counts.
add_quality_check("baseline_recommendation_snapshot", "row_count_positive", "PASS" if len(baseline_recommendation_snapshot) > 0 else "FAIL", len(baseline_recommendation_snapshot), ">0")
add_quality_check("current_auth_history", "row_count_positive", "PASS" if len(current_auth_history) > 0 else "FAIL", len(current_auth_history), ">0")

row_count_ratio = len(current_auth_history) / max(len(baseline_auth_history_profile), 1)
row_status = "PASS" if ROW_COUNT_RATIO_WARNING_LOW <= row_count_ratio <= ROW_COUNT_RATIO_WARNING_HIGH else "WARNING"
add_quality_check(
    "baseline_vs_current",
    "row_count_ratio_within_expected_range",
    row_status,
    observed_value=row_count_ratio,
    threshold_value=f"[{ROW_COUNT_RATIO_WARNING_LOW}, {ROW_COUNT_RATIO_WARNING_HIGH}]",
    details="Ratio = current_auth_history_rows / baseline_auth_history_rows. This is structural only; new combos are handled in recommendation_coverage_status.",
)

# Uniqueness by grain.
def duplicate_rate(df, keys):
    if not set(keys).issubset(df.columns):
        return np.nan
    return float(df.duplicated(keys).mean())

baseline_dup_rate = duplicate_rate(baseline_recommendation_snapshot, KEY_COLUMNS)
current_dup_rate = duplicate_rate(current_auth_history, KEY_COLUMNS)
add_quality_check("baseline_recommendation_snapshot", "grain_uniqueness", "PASS" if baseline_dup_rate == 0 else "FAIL", baseline_dup_rate, 0)
add_quality_check("current_auth_history", "grain_uniqueness", "PASS" if current_dup_rate == 0 else "FAIL", current_dup_rate, 0)

# Price monotonicity.
def monotonic_rate(df, cols):
    cols = [c for c in cols if c in df.columns]
    if len(cols) < 2:
        return np.nan
    values = df[cols].apply(pd.to_numeric, errors="coerce")
    valid_rows = values.notna().all(axis=1)
    if valid_rows.sum() == 0:
        return np.nan
    diffs = values.loc[valid_rows].diff(axis=1).iloc[:, 1:]
    return float((diffs >= -1e-12).all(axis=1).mean())

for dataset_name, df in [("baseline_auth_history", baseline_auth_history_profile), ("current_auth_history", current_auth_history)]:
    rate = monotonic_rate(df, ["P0_PRICE", "P20_PRICE", "P50_PRICE", "P85_PRICE", "P100_PRICE"])
    status = "PASS" if pd.isna(rate) or rate >= 0.99 else "FAIL"
    add_quality_check(dataset_name, "price_percentiles_monotonic", status, rate, ">=0.99")

# Non-negative prices.
for dataset_name, df in [("baseline_auth_history", baseline_auth_history_profile), ("current_auth_history", current_auth_history)]:
    price_cols = [c for c in ["P20_PRICE", "P50_PRICE", "P85_PRICE"] if c in df.columns]
    if price_cols:
        min_price = df[price_cols].apply(pd.to_numeric, errors="coerce").min().min()
        status = "PASS" if pd.isna(min_price) or min_price >= 0 else "FAIL"
        add_quality_check(dataset_name, "non_negative_core_prices", status, min_price, ">=0")

# Null rates in critical columns.
for dataset_name, df, cols in [
    ("baseline_recommendation_snapshot", baseline_recommendation_snapshot, ["baseline_recommended_price"]),
    ("current_auth_history", current_auth_history, ["P20_PRICE", "P50_PRICE", "P85_PRICE"]),
]:
    for col in cols:
        if col in df.columns:
            null_rate = float(df[col].isna().mean())
            status = "PASS" if null_rate <= 0.01 else "WARNING" if null_rate <= 0.05 else "FAIL"
            add_quality_check(dataset_name, f"null_rate_{col}", status, null_rate, "<=0.01 PASS; <=0.05 WARNING")

# Coverage of baseline recommendations in current history.
baseline_keys = set(map(tuple, baseline_recommendation_snapshot[KEY_COLUMNS].astype(str).to_numpy()))
current_keys = set(map(tuple, current_auth_history[KEY_COLUMNS].astype(str).to_numpy()))
matched_keys = baseline_keys.intersection(current_keys)
coverage_rate = len(matched_keys) / max(len(baseline_keys), 1)

if coverage_rate < CURRENT_HISTORY_COVERAGE_RED_THRESHOLD:
    coverage_status = "Red"
elif coverage_rate < CURRENT_HISTORY_COVERAGE_YELLOW_THRESHOLD:
    coverage_status = "Yellow"
else:
    coverage_status = "Green"

add_coverage_check(
    "current_history_coverage_of_baseline_recommendations",
    coverage_status,
    observed_value=coverage_rate,
    threshold_value=f">={CURRENT_HISTORY_COVERAGE_YELLOW_THRESHOLD} Green; <{CURRENT_HISTORY_COVERAGE_RED_THRESHOLD} Red",
    details="Share of baseline recommendation keys found in current AUTH history.",
)

# New combo rate is not data quality; it is recommendation coverage / scoring coverage.
new_combo_rate = len(current_keys - baseline_keys) / max(len(current_keys), 1)
if new_combo_rate >= NEW_COMBO_RATE_RED_THRESHOLD:
    new_combo_status = "Red"
elif new_combo_rate >= NEW_COMBO_RATE_YELLOW_THRESHOLD:
    new_combo_status = "Yellow"
else:
    new_combo_status = "Green"

add_coverage_check(
    "new_combo_rate_without_baseline_recommendation",
    new_combo_status,
    observed_value=new_combo_rate,
    threshold_value=f"Yellow>={NEW_COMBO_RATE_YELLOW_THRESHOLD}; Red>={NEW_COMBO_RATE_RED_THRESHOLD}",
    details="Share of current AUTH combos without baseline recommendation. This justifies scoring/coverage update, not data-quality remediation.",
)

# Bin catalog metadata available.
add_quality_check(
    "monitoring_configuration",
    "bin_catalog_scope_declared",
    "PASS" if BIN_CATALOG_SCOPE == "AUTH_AND_NON_AUTH_GLOBAL" else "WARNING",
    observed_value=BIN_CATALOG_SCOPE,
    threshold_value="AUTH_AND_NON_AUTH_GLOBAL",
    details="Bins are monitored over AUTH rows but catalog was created globally with AUTH + non-AUTH data.",
)

# Catalog/bin coverage over current AUTH snapshot. This is not price drift.
INPUT_BIN_MODE_COLUMNS = [
    "order_size_bin_mode",
    "disty_margin_original_bin_mode",
    "channel_margin_share_bin_mode",
    "q0_local_obs_bin",
    "revenue_weight_bucket",
]
for col in INPUT_BIN_MODE_COLUMNS:
    if col in current_auth_history.columns:
        s = current_auth_history[col].fillna("Missing").astype(str)
        missing_rate = float(s.isin(["Missing", "__MISSING__", "nan", "None", ""]).mean())
        add_catalog_coverage_check(col, missing_rate, details="Missing means the current AUTH snapshot did not receive a valid bin/category value from the fixed global catalog or preparation step.")

# Dataframes.
data_quality_log = pd.DataFrame(quality_records)
recommendation_coverage_log = pd.DataFrame(coverage_records)
catalog_bin_coverage_log = pd.DataFrame(catalog_coverage_records)

quality_status_summary = (
    data_quality_log.groupby(["dataset_name", "check_status"])
    .size()
    .reset_index(name="n_checks")
)

recommendation_coverage_summary = (
    recommendation_coverage_log.groupby(["check_status"])
    .size()
    .reset_index(name="n_checks")
)

catalog_bin_coverage_summary = (
    catalog_bin_coverage_log.groupby(["coverage_status"])
    .size()
    .reset_index(name="n_checks")
    if not catalog_bin_coverage_log.empty else pd.DataFrame()
)

print("Data quality log:")
display(data_quality_log)
print("Recommendation coverage log:")
display(recommendation_coverage_log)
print("Catalog/bin coverage log:")
display(catalog_bin_coverage_log)
print("Quality status summary:")
display(quality_status_summary)


## 6. Métricas de drift para historia AUTH

Esta sección compara la historia AUTH baseline contra la historia AUTH actual simulada. No usa recomendaciones nuevas.

El objetivo es saber si cambió la población o comportamiento histórico antes de decidir mantener/revisar/correr modelo.


### 6.0 Lógica de drift compartida

Las funciones PSI/KS del notebook fueron migradas a `pricing.auth_monitoring.rules.auth_history_drift` para que el pipeline ejecute el mismo contrato desde el step `calculate_auth_history_drift`.


In [ ]:

# =========================================================
# 6.1 Calculate AUTH history drift log
# =========================================================

drift_result = calculate_auth_history_drift(
    baseline_auth_history_profile=baseline_auth_history_profile.replace({np.nan: None}).to_dict(orient="records"),
    current_auth_history=current_auth_history.replace({np.nan: None}).to_dict(orient="records"),
    run_id=DRIFT_RUN_ID,
    config=MONITORING_CONFIG,
)
input_history_drift_log = pd.DataFrame(drift_result.drift_log)

drift_status_summary = (
    input_history_drift_log.groupby(["drift_metric", "drift_status"], dropna=False)
    .size()
    .reset_index(name="n_metrics")
    if not input_history_drift_log.empty
    else pd.DataFrame(columns=["drift_metric", "drift_status", "n_metrics"])
)

print("AUTH history drift log:")
display(input_history_drift_log)
print("Drift status summary:")
display(drift_status_summary)


## 7. Vigencia de recomendación contra historia AUTH actual

Esta sección evalúa la recomendación vigente contra la historia AUTH actual real.

Cambios clave de la versión v3:

- Solo calcula posición P20/P85 cuando la banda es válida.
- Si `P20 = P85`, el caso se clasifica como `Collapsed_Band` / `Single_Price_Regime` y se evalúa contra P50, no como extremo.
- Compara posición baseline vs posición current para saber si el modelo ya había recomendado en un extremo (`Persistent_Edge`) o si la historia actual movió la recomendación hacia el extremo (`Moved_To_Edge`).
- Introduce estado `Watch` para señales de vigilancia que no deberían inflar Yellow accionable.


In [ ]:

# =========================================================
# 7. Recommendation validity vs current AUTH history
# =========================================================

# La logica de esta seccion ya vive en pricing.auth_monitoring.rules.recommendation_validity.
# La copia de transicion conserva DataFrames con los mismos nombres para que las celdas
# exploratorias y visualizaciones sigan funcionando mientras el pipeline usa componentes.
validity_result = calculate_recommendation_validity(
    baseline_recommendation_snapshot=baseline_recommendation_snapshot.replace({np.nan: None}).to_dict(orient="records"),
    baseline_auth_history_profile=baseline_auth_history_profile.replace({np.nan: None}).to_dict(orient="records"),
    current_auth_history=current_auth_history.replace({np.nan: None}).to_dict(orient="records"),
    config=MONITORING_CONFIG,
)

validity_df = pd.DataFrame(validity_result.validity_log)
new_combo_log = pd.DataFrame(validity_result.new_combo_log)
validity_summary = pd.DataFrame(validity_result.validity_summary)
validity_revenue_summary = pd.DataFrame(validity_result.validity_revenue_summary)
validity_reason_summary = pd.DataFrame(validity_result.validity_reason_summary)

if not validity_df.empty and "edge_transition_type" in validity_df.columns:
    edge_transition_summary = (
        validity_df.groupby("edge_transition_type", dropna=False)
        .agg(
            n_recommendations=("baseline_recommended_price", "size"),
            baseline_revenue_sum=("baseline_revenue_sum_numeric", "sum"),
        )
        .reset_index()
        .sort_values("n_recommendations", ascending=False)
    )
else:
    edge_transition_summary = pd.DataFrame()

if not validity_df.empty and "current_band_state" in validity_df.columns:
    band_state_summary = (
        validity_df.groupby("current_band_state", dropna=False)
        .agg(
            n_recommendations=("baseline_recommended_price", "size"),
            baseline_revenue_sum=("baseline_revenue_sum_numeric", "sum"),
        )
        .reset_index()
        .sort_values("n_recommendations", ascending=False)
    )
else:
    band_state_summary = pd.DataFrame()

# Sensitivity profiles remain a notebook-only exploratory output until that section is extracted.
sensitivity_profile_summary = pd.DataFrame()

print("Recommendation validity summary:")
display(validity_summary)
print("Revenue-weighted validity summary:")
display(validity_revenue_summary)
print("Band state summary:")
display(band_state_summary)
print("Edge transition summary:")
display(edge_transition_summary)
print("Validity reason summary:")
display(validity_reason_summary.head(30) if not validity_reason_summary.empty else validity_reason_summary)
print("New combos without baseline recommendation:", len(new_combo_log))
display(validity_df.head())


## 7.1 Política operativa para nuevos combos y decisión scoring vs retraining

Esta sección agrega una capa de decisión MLOps más fina:

- **Scoring/update**: generar nuevas recomendaciones con el pipeline de optimización usando elasticidad HB-SVI existente, heredada o default `-1`.
- **HB-SVI retraining**: reentrenar el modelo jerárquico solo cuando haya evidencia suficiente de drift/model health o nueva señal estadística.
- **Catalog review**: revisar bins/catálogos cuando los datos actuales dejan de caer bien en el catálogo fijo.
- **Review**: mandar a Pricing/PM solo casos Yellow/Red accionables.

Esto evita interpretar `new_combo_rate` como falla del modelo. Los nuevos combos suelen justificar una actualización operativa de recomendaciones, no necesariamente un reentrenamiento completo.


In [ ]:

# =========================================================
# 7.1 Operational policy: new-combo scoring, catalog review and retraining separation
# =========================================================

# La decision operativa y la separacion scoring/update vs retraining ya se calculan en
# pricing.auth_monitoring.rules.operational_decision. Esta celda conserva los resúmenes de apoyo
# que usa el reporte narrativo mientras esa salida termina de migrarse a componentes.
current_revenue_total = pd.to_numeric(current_auth_history.get("revenue_sum", pd.Series(dtype=float)), errors="coerce").sum()
new_combo_revenue_sum = (
    pd.to_numeric(new_combo_log.get("revenue_sum", pd.Series(dtype=float)), errors="coerce").sum()
    if not new_combo_log.empty
    else 0.0
)
new_combo_revenue_share = float(new_combo_revenue_sum / current_revenue_total) if current_revenue_total > 0 else np.nan

new_combo_scoring_policy_summary = pd.DataFrame()
new_combo_operational_summary = pd.DataFrame([{
    "drift_run_id": DRIFT_RUN_ID,
    "new_combo_count": int(len(new_combo_log)),
    "new_combo_rate": float(new_combo_rate),
    "new_combo_revenue_sum": float(new_combo_revenue_sum) if pd.notna(new_combo_revenue_sum) else np.nan,
    "new_combo_revenue_share_current": new_combo_revenue_share,
    "recommended_new_combo_action": (
        "RUN_SCORING_UPDATE_FOR_NEW_COMBOS"
        if (new_combo_rate >= SCORING_UPDATE_NEW_COMBO_RATE_THRESHOLD or len(new_combo_log) >= SCORING_UPDATE_NEW_COMBO_COUNT_THRESHOLD)
        else "WATCH_NEW_COMBO_COVERAGE"
    ),
    "interpretation": "New combos are a recommendation coverage trigger. They do not imply HB-SVI failure by themselves.",
}])

PRICE_DRIFT_VARIABLES = ["P20_PRICE", "P50_PRICE", "P85_PRICE"]
price_drift_log = input_history_drift_log[
    input_history_drift_log["variable_name"].isin(PRICE_DRIFT_VARIABLES)
    & input_history_drift_log["drift_metric"].isin(["PSI", "KS"])
].copy()

existing_recommendation_risk_summary = pd.DataFrame([{
    "red_recommendation_rate": float((validity_df["auth_recommendation_validity_status"] == "Red").mean()),
    "yellow_recommendation_rate": float((validity_df["auth_recommendation_validity_status"] == "Yellow").mean()),
    "watch_recommendation_rate": float((validity_df["auth_recommendation_validity_status"] == "Watch").mean()),
    "red_revenue_share": float(validity_revenue_summary.loc[
        validity_revenue_summary["auth_recommendation_validity_status"].eq("Red"),
        "baseline_revenue_share"
    ].sum()) if not validity_revenue_summary.empty else np.nan,
    "yellow_revenue_share": float(validity_revenue_summary.loc[
        validity_revenue_summary["auth_recommendation_validity_status"].eq("Yellow"),
        "baseline_revenue_share"
    ].sum()) if not validity_revenue_summary.empty else np.nan,
}])

print("New combo operational summary:")
display(new_combo_operational_summary)
print("New combo scoring policy summary:")
display(new_combo_scoring_policy_summary)
print("Price drift signal log:")
display(price_drift_log)


## 8. Semáforos finales y decisión de corrida

Los estados globales quedan separados:

- `data_quality_status`
- `recommendation_coverage_status`
- `catalog_bin_coverage_status`
- `auth_history_drift_status`
- `recommendation_validity_global_status`
- `run_readiness_status`

Esto evita confundir nuevos combos o cobertura de catálogo con defectos estructurales de datos.


In [ ]:

# =========================================================
# 8. Final statuses and operational decision layer
# =========================================================

# La compuerta operacional ya vive en pricing.auth_monitoring.rules.operational_decision y es
# el contrato que usa el step calculate_operational_decision del pipeline.
operational_result = calculate_operational_decision(
    run_id=DRIFT_RUN_ID,
    validity_log=validity_df.replace({np.nan: None}).to_dict(orient="records"),
    validity_revenue_summary=validity_revenue_summary.replace({np.nan: None}).to_dict(orient="records"),
    new_combo_log=new_combo_log.replace({np.nan: None}).to_dict(orient="records"),
    data_quality_log=data_quality_log.replace({np.nan: None}).to_dict(orient="records"),
    recommendation_coverage_log=recommendation_coverage_log.replace({np.nan: None}).to_dict(orient="records"),
    catalog_bin_coverage_log=catalog_bin_coverage_log.replace({np.nan: None}).to_dict(orient="records"),
    input_history_drift_log=input_history_drift_log.replace({np.nan: None}).to_dict(orient="records"),
    config=MONITORING_CONFIG,
)

run_readiness_summary = dict(operational_result.run_readiness_summary)
operational_decision_record = dict(operational_result.operational_decision_summary)
run_readiness_summary_df = pd.DataFrame([run_readiness_summary])
operational_decision_summary = pd.DataFrame([operational_decision_record])

# Variables preservadas para visualizaciones y reporte narrativo posteriores.
data_quality_status = run_readiness_summary["data_quality_status"]
recommendation_coverage_status = run_readiness_summary["recommendation_coverage_status"]
catalog_bin_coverage_status = run_readiness_summary["catalog_bin_coverage_status"]
auth_history_drift_status = run_readiness_summary["auth_history_drift_status"]
price_drift_status = run_readiness_summary["price_drift_status"]
recommendation_validity_global_status = run_readiness_summary["recommendation_validity_global_status"]
run_readiness_status = run_readiness_summary["run_readiness_status"]
recommended_operational_action = run_readiness_summary["recommended_operational_action"]
scoring_update_recommendation = run_readiness_summary["scoring_update_recommendation"]
hb_svi_retraining_recommendation = run_readiness_summary["hb_svi_retraining_recommendation"]
catalog_rebaseline_recommendation = run_readiness_summary["catalog_rebaseline_recommendation"]
run_readiness_decision = run_readiness_summary["run_readiness_decision"]

display(run_readiness_summary_df)
display(operational_decision_summary)


## 9. Segmentos y casos principales para revisión

Esta sección identifica dónde se concentran los casos Red/Yellow, usando la granularidad de negocio disponible.


In [ ]:
# =========================================================
# 9. Segment-level summaries and review queues
# =========================================================

segment_columns = [c for c in ["baseline_custombusinessgroup_mode", "vpareadescription", "distysegment"] if c in validity_df.columns]
if "baseline_custombusinessgroup_mode" not in segment_columns and "custombusinessgroup_mode" in validity_df.columns:
    segment_columns = ["custombusinessgroup_mode", "vpareadescription", "distysegment"]

if segment_columns:
    segment_validity_summary = (
        validity_df
        .groupby(segment_columns, dropna=False)
        .agg(
            n_recommendations=("baseline_recommended_price", "size"),
            baseline_revenue_sum=("baseline_revenue_sum_numeric", "sum"),
            green_rate=("auth_recommendation_validity_status", lambda s: (s == "Green").mean()),
            watch_rate=("auth_recommendation_validity_status", lambda s: (s == "Watch").mean()),
            yellow_rate=("auth_recommendation_validity_status", lambda s: (s == "Yellow").mean()),
            red_rate=("auth_recommendation_validity_status", lambda s: (s == "Red").mean()),
            not_evaluable_rate=("auth_recommendation_validity_status", lambda s: (s == "Not_Evaluable").mean()),
            avg_gap_vs_current_auth_p50_pct=("recommendation_gap_vs_current_auth_p50_pct", "mean"),
            avg_current_auth_p50_shift_vs_baseline_pct=("current_auth_p50_shift_vs_baseline_pct", "mean"),
            valid_band_rate=("current_band_state", lambda s: (s == "Valid_Band").mean()),
            collapsed_band_rate=("current_band_state", lambda s: (s == "Collapsed_Band").mean()),
            low_support_band_rate=("current_band_state", lambda s: (s == "Low_Support_Band").mean()),
            moved_to_edge_rate=("edge_transition_type", lambda s: (s == "Moved_To_Edge").mean()),
            persistent_edge_rate=("edge_transition_type", lambda s: (s == "Persistent_Edge").mean()),
        )
        .reset_index()
    )
    segment_validity_summary["baseline_revenue_share"] = segment_validity_summary["baseline_revenue_sum"] / max(segment_validity_summary["baseline_revenue_sum"].sum(), 1e-12)
    segment_validity_summary = segment_validity_summary.sort_values(["red_rate", "yellow_rate", "baseline_revenue_sum"], ascending=[False, False, False])
else:
    segment_validity_summary = pd.DataFrame()

# Top review cases. Watch is included after Red/Yellow but below actionable alerts.
status_rank = {"Red": 4, "Yellow": 3, "Watch": 2, "Not_Evaluable": 1, "Green": 0}
validity_df["_status_rank"] = validity_df["auth_recommendation_validity_status"].map(status_rank).fillna(0)

top_recommendation_review_cases = (
    validity_df
    .sort_values(["_status_rank", "baseline_revenue_sum_numeric"], ascending=[False, False])
    .head(150)
    .drop(columns=["_status_rank"], errors="ignore")
)

print("Segment validity summary:")
display(segment_validity_summary.head(30))
print("Top recommendation review cases:")
review_cols = KEY_COLUMNS + [
    "baseline_recommended_price", "baseline_P20_PRICE", "baseline_P50_PRICE", "baseline_P85_PRICE",
    "current_P20_PRICE", "current_P50_PRICE", "current_P85_PRICE",
    "baseline_band_state", "current_band_state",
    "recommendation_position_baseline_auth_band", "recommendation_position_current_auth_band",
    "recommendation_position_shift_current_vs_baseline",
    "edge_transition_type", "recommendation_gap_vs_current_auth_p50_pct",
    "current_auth_p50_shift_vs_baseline_pct", "auth_recommendation_validity_status",
    "auth_recommendation_validity_reason", "recommended_action", "baseline_revenue_sum_numeric"
]
review_cols = [c for c in review_cols if c in top_recommendation_review_cases.columns]
display(top_recommendation_review_cases[review_cols].head(30))

# Add minimum support interpretation for dashboard usage.
if not segment_validity_summary.empty:
    segment_validity_summary["segment_support_status"] = np.where(
        segment_validity_summary["n_recommendations"] >= MIN_SEGMENT_SUPPORT_FOR_DASHBOARD,
        "Sufficient_Support",
        "Low_Support"
    )
    segment_validity_summary["dashboard_include_flag"] = (
        segment_validity_summary["n_recommendations"] >= MIN_SEGMENT_SUPPORT_FOR_DASHBOARD
    )
    # Score prioritizes actionable Red/Yellow over Watch.
    segment_validity_summary["segment_review_priority_score"] = (
        100 * segment_validity_summary["baseline_revenue_share"].fillna(0)
        + 70 * segment_validity_summary["red_rate"].fillna(0)
        + 35 * segment_validity_summary["yellow_rate"].fillna(0)
        + 10 * segment_validity_summary["watch_rate"].fillna(0)
        + 15 * segment_validity_summary["moved_to_edge_rate"].fillna(0)
    )

    segment_dashboard_summary = (
        segment_validity_summary
        .query("dashboard_include_flag == True")
        .sort_values(["segment_review_priority_score", "baseline_revenue_sum"], ascending=[False, False])
        .copy()
    )
else:
    segment_dashboard_summary = pd.DataFrame()

# Executive view for PM / Pricing review.
executive_review_columns = [
    "vpareadescription", "distysegment", "baseline_custombusinessgroup_mode",
    "n_recommendations", "baseline_revenue_share", "red_rate", "yellow_rate", "watch_rate", "green_rate",
    "not_evaluable_rate", "valid_band_rate", "collapsed_band_rate", "low_support_band_rate",
    "moved_to_edge_rate", "persistent_edge_rate",
    "avg_gap_vs_current_auth_p50_pct", "avg_current_auth_p50_shift_vs_baseline_pct",
    "segment_support_status", "segment_review_priority_score"
]
executive_review_columns = [c for c in executive_review_columns if c in segment_dashboard_summary.columns]

print("Segment dashboard summary, sufficient support only:")
display(segment_dashboard_summary[executive_review_columns].head(15))


### 9.1 Nota sobre soporte por segmento

Las tasas por segmento pueden ser engañosas cuando el segmento tiene pocas observaciones. Por eso se agrega `segment_support_status`:

- `Sufficient_Support`: al menos 30 recomendaciones en el segmento.
- `Low_Support`: menos de 30 recomendaciones; el segmento se conserva en la evidencia, pero no debe ser el principal motor de decisión ejecutiva.

Para dashboards de negocio se recomienda priorizar `segment_dashboard_summary`, que filtra segmentos con soporte suficiente y ordena por una combinación de revenue share, red rate y yellow rate.


## 10. Drift de bins globales sobre registros AUTH

Esta sección compara la distribución de registros AUTH por bins. Los bins fueron creados usando AUTH + no AUTH, pero aquí se monitorea cómo se comporta la población AUTH dentro de esos bins.


In [ ]:
# =========================================================
# 10. Global bin drift over AUTH records
# =========================================================

bin_drift_records = []

for col in bin_cols_available:
    if col not in baseline_auth_history_profile.columns or col not in current_auth_history.columns:
        continue
    base_dist = baseline_auth_history_profile[col].fillna("__MISSING__").astype(str).value_counts(normalize=True)
    curr_dist = current_auth_history[col].fillna("__MISSING__").astype(str).value_counts(normalize=True)
    all_bins = sorted(set(base_dist.index).union(set(curr_dist.index)))
    for bin_value in all_bins:
        baseline_pct = float(base_dist.get(bin_value, 0.0))
        current_pct = float(curr_dist.get(bin_value, 0.0))
        bin_drift_records.append({
            "drift_run_id": DRIFT_RUN_ID,
            "monitoring_scope": MONITORING_SCOPE,
            "bin_catalog_scope": BIN_CATALOG_SCOPE,
            "bin_catalog_version": BIN_CATALOG_VERSION,
            "column_name": col,
            "bin_value": bin_value,
            "baseline_pct": baseline_pct,
            "current_pct": current_pct,
            "pct_point_change": current_pct - baseline_pct,
            "abs_pct_point_change": abs(current_pct - baseline_pct),
        })

bin_drift_summary = pd.DataFrame(bin_drift_records)
if not bin_drift_summary.empty:
    bin_drift_summary = bin_drift_summary.sort_values("abs_pct_point_change", ascending=False)

display(bin_drift_summary.head(40))


## 11. Visualizaciones

Las gráficas ahora separan:

- Conteo y revenue por estado (`Green`, `Watch`, `Yellow`, `Red`, `Not_Evaluable`).
- Estado de banda actual (`Valid_Band`, `Collapsed_Band`, `Low_Support_Band`).
- Transición de extremos baseline vs current.
- Gap contra P50 con umbrales del perfil seleccionado.
- Sensibilidad por perfil de alerta.


### 11.0 Guía de lectura de visualizaciones

- Los histogramas usan `bins=30` únicamente para dibujar las distribuciones; no significa que se analicen solo 30 datos. Las métricas se calculan sobre todas las filas disponibles.
- La posición de recomendación usa la escala `0 = current AUTH P20` y `1 = current AUTH P85`. Valores menores a 0 indican recomendación debajo del P20 actual; valores mayores a 1 indican recomendación arriba del P85 actual.
- Las métricas de cobertura no deben compararse directamente con PSI o KS. Por eso el notebook separa las gráficas de drift estadístico y cobertura.
- Los bins de negocio se monitorean como distribución de registros AUTH dentro de un catálogo global creado con AUTH + no-AUTH.


In [ ]:
# =========================================================
# 11. Visualizations
# =========================================================

figures_manifest = []

def save_current_figure(filename, title):
    path = OUTPUT_DIRS["figures"] / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    figures_manifest.append({
        "figure_name": filename,
        "figure_title": title,
        "figure_path": str(path),
        "drift_run_id": DRIFT_RUN_ID,
    })
    plt.show()

STATUS_ORDER = ["Green", "Watch", "Yellow", "Red", "Not_Evaluable"]

# Status counts.
status_counts = validity_df["auth_recommendation_validity_status"].value_counts().reindex(STATUS_ORDER, fill_value=0)
plt.figure(figsize=(8, 5))
status_counts.plot(kind="bar")
plt.title("AUTH recommendation validity status")
plt.xlabel("Status")
plt.ylabel("Number of recommendations")
plt.xticks(rotation=0)
save_current_figure("auth_recommendation_validity_status.png", "AUTH recommendation validity status")

# Revenue share by status.
rev_plot = validity_revenue_summary.set_index("auth_recommendation_validity_status").reindex(STATUS_ORDER).fillna(0)
plt.figure(figsize=(8, 5))
rev_plot["baseline_revenue_share"].plot(kind="bar")
plt.title("Baseline revenue share by recommendation validity status")
plt.xlabel("Status")
plt.ylabel("Revenue share")
plt.xticks(rotation=0)
save_current_figure("auth_recommendation_validity_revenue_share.png", "Baseline revenue share by recommendation validity status")

# Current band state.
if not band_state_summary.empty:
    band_counts = validity_df["current_band_state"].value_counts()
    plt.figure(figsize=(9, 5))
    band_counts.plot(kind="bar")
    plt.title("Current AUTH band state")
    plt.xlabel("Band state")
    plt.ylabel("Number of recommendations")
    plt.xticks(rotation=30, ha="right")
    save_current_figure("current_auth_band_state.png", "Current AUTH band state")

# Edge transition.
if "edge_transition_type" in validity_df.columns:
    edge_counts = validity_df["edge_transition_type"].value_counts().head(15)
    plt.figure(figsize=(10, 5))
    edge_counts.plot(kind="bar")
    plt.title("Baseline-to-current edge transition type")
    plt.xlabel("Edge transition")
    plt.ylabel("Number of recommendations")
    plt.xticks(rotation=35, ha="right")
    save_current_figure("edge_transition_type.png", "Baseline-to-current edge transition type")

# Recommendation position distribution for valid bands only.
valid_position = validity_df.loc[validity_df["current_band_state"].eq("Valid_Band"), "recommendation_position_current_auth_band"].replace([np.inf, -np.inf], np.nan).dropna()
if not valid_position.empty:
    plt.figure(figsize=(9, 5))
    valid_position.clip(-1, 2).hist(bins=30)
    plt.axvline(0, linestyle="--")
    plt.axvline(1, linestyle="--")
    plt.axvline(NEAR_BAND_EDGE_THRESHOLD, linestyle=":")
    plt.axvline(1 - NEAR_BAND_EDGE_THRESHOLD, linestyle=":")
    plt.title("Recommendation position in current AUTH P20-P85 band — valid bands only")
    plt.xlabel("Position: 0=P20, 1=P85")
    plt.ylabel("Frequency")
    save_current_figure("recommendation_position_current_auth_band_valid_only.png", "Recommendation position in current AUTH band, valid bands only")

# Position shift baseline vs current.
if "recommendation_position_shift_current_vs_baseline" in validity_df.columns:
    position_shift = validity_df["recommendation_position_shift_current_vs_baseline"].replace([np.inf, -np.inf], np.nan).dropna()
    if not position_shift.empty:
        plt.figure(figsize=(9, 5))
        position_shift.clip(-2, 2).hist(bins=30)
        plt.axvline(0, linestyle="--")
        plt.title("Recommendation position shift: current band vs baseline band")
        plt.xlabel("Current position - baseline position")
        plt.ylabel("Frequency")
        save_current_figure("recommendation_position_shift_current_vs_baseline.png", "Recommendation position shift current vs baseline")

# Gap vs current P50.
if "recommendation_gap_vs_current_auth_p50_pct" in validity_df.columns:
    plt.figure(figsize=(9, 5))
    validity_df["recommendation_gap_vs_current_auth_p50_pct"].replace([np.inf, -np.inf], np.nan).dropna().clip(-1, 1).hist(bins=30)
    plt.axvline(0, linestyle="--")
    plt.axvline(GAP_VS_P50_YELLOW, linestyle=":")
    plt.axvline(-GAP_VS_P50_YELLOW, linestyle=":")
    plt.axvline(GAP_VS_P50_RED, linestyle="--")
    plt.axvline(-GAP_VS_P50_RED, linestyle="--")
    plt.title("Recommendation gap vs current AUTH P50 (%)")
    plt.xlabel("Gap vs current AUTH P50")
    plt.ylabel("Frequency")
    save_current_figure("recommendation_gap_vs_current_auth_p50_pct.png", "Recommendation gap vs current AUTH P50")

# Current P50 shift vs baseline P50.
# If all values are effectively zero, show a clear text card instead of an empty-looking histogram.
if "current_auth_p50_shift_vs_baseline_pct" in validity_df.columns:
    x_p50_shift = validity_df["current_auth_p50_shift_vs_baseline_pct"].replace([np.inf, -np.inf], np.nan).dropna()
    if not x_p50_shift.empty:
        if x_p50_shift.abs().max() < 1e-6:
            plt.figure(figsize=(9, 4))
            plt.text(
                0.5, 0.5,
                f"No measurable P50 shift for baseline-covered recommendations\n"
                f"n = {len(x_p50_shift):,}\n"
                f"max abs shift = {x_p50_shift.abs().max():.6%}",
                ha="center",
                va="center",
                fontsize=12,
            )
            plt.axis("off")
            save_current_figure("current_auth_p50_shift_vs_baseline_pct_no_shift.png", "Current AUTH P50 shift vs baseline P50 - no measurable shift")
        else:
            plt.figure(figsize=(9, 5))
            x_p50_shift.clip(-0.50, 0.50).hist(bins=40)
            plt.axvline(0, linestyle="--")
            plt.axvline(P50_SHIFT_YELLOW, linestyle=":")
            plt.axvline(-P50_SHIFT_YELLOW, linestyle=":")
            plt.title(f"Current AUTH P50 shift vs baseline P50 | n={len(x_p50_shift):,}")
            plt.xlabel("Current P50 shift vs baseline P50")
            plt.ylabel("Frequency")
            save_current_figure("current_auth_p50_shift_vs_baseline_pct.png", "Current AUTH P50 shift vs baseline P50")

# Alert reasons.
if not validity_reason_summary.empty:
    top_reasons = validity_reason_summary.sort_values("n_recommendations", ascending=False).head(15)
    plt.figure(figsize=(11, 6))
    labels = top_reasons["auth_recommendation_validity_status"] + " | " + top_reasons["auth_recommendation_validity_reason"]
    plt.barh(labels, top_reasons["n_recommendations"])
    plt.title("Top recommendation validity reasons")
    plt.xlabel("Number of recommendations")
    plt.gca().invert_yaxis()
    save_current_figure("top_recommendation_validity_reasons.png", "Top recommendation validity reasons")

# Sensitivity profiles.
if not sensitivity_profile_summary.empty:
    for metric in ["recommendation_share", "baseline_revenue_share"]:
        pivot = sensitivity_profile_summary.pivot(index="alert_profile", columns="status", values=metric).reindex(columns=STATUS_ORDER).fillna(0)
        plt.figure(figsize=(9, 5))
        bottom = np.zeros(len(pivot))
        x = np.arange(len(pivot))
        for status in STATUS_ORDER:
            vals = pivot[status].values
            plt.bar(x, vals, bottom=bottom, label=status)
            bottom += vals
        plt.xticks(x, pivot.index)
        plt.ylim(0, 1)
        plt.title(f"Sensitivity profiles by {metric}")
        plt.ylabel(metric)
        plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
        save_current_figure(f"sensitivity_profiles_{metric}.png", f"Sensitivity profiles by {metric}")

# History drift top metrics.
statistical_history_drift = input_history_drift_log[
    input_history_drift_log["drift_metric"].isin(["PSI", "KS", "Categorical_PSI"])
].dropna(subset=["drift_value"]).sort_values("drift_value", ascending=False).head(15)

if not statistical_history_drift.empty:
    plt.figure(figsize=(11, 6))
    labels = statistical_history_drift["variable_name"] + " | " + statistical_history_drift["drift_metric"]
    plt.barh(labels, statistical_history_drift["drift_value"])
    plt.title("Top statistical AUTH history drift metrics")
    plt.xlabel("Drift value: PSI / KS / categorical PSI")
    plt.gca().invert_yaxis()
    save_current_figure("top_statistical_auth_history_drift_metrics.png", "Top statistical AUTH history drift metrics")

coverage_history_metrics = input_history_drift_log[
    input_history_drift_log["drift_metric"] == "Coverage_Rate"
].copy()

if not coverage_history_metrics.empty:
    plt.figure(figsize=(9, 4))
    labels = coverage_history_metrics["variable_name"]
    plt.barh(labels, coverage_history_metrics["drift_value"])
    plt.axvline(CURRENT_HISTORY_COVERAGE_YELLOW_THRESHOLD, linestyle="--")
    plt.axvline(CURRENT_HISTORY_COVERAGE_RED_THRESHOLD, linestyle="--")
    plt.title("AUTH coverage metrics")
    plt.xlabel("Coverage / new-combo rate value")
    plt.gca().invert_yaxis()
    save_current_figure("auth_coverage_metrics.png", "AUTH coverage metrics")

# Bin drift top changes.
if not bin_drift_summary.empty:
    top_bins = bin_drift_summary.head(15).copy()
    plt.figure(figsize=(11, 6))
    labels = top_bins["column_name"] + " = " + top_bins["bin_value"]
    plt.barh(labels, top_bins["pct_point_change"])
    plt.title("Top global bin distribution changes over AUTH records")
    plt.xlabel("Percentage point change")
    plt.gca().invert_yaxis()
    save_current_figure("top_global_bin_changes_auth_scope.png", "Top global bin changes over AUTH records")


# New combo scoring policy.
if "new_combo_scoring_policy_summary" in globals() and not new_combo_scoring_policy_summary.empty:
    policy_plot = new_combo_scoring_policy_summary.sort_values("n_new_combos", ascending=True)
    labels = policy_plot["scoring_policy"] + " | " + policy_plot["elasticity_confidence"]
    plt.figure(figsize=(11, 5))
    plt.barh(labels, policy_plot["n_new_combos"])
    plt.title("New combo scoring policy")
    plt.xlabel("Number of new combos")
    save_current_figure("new_combo_scoring_policy.png", "New combo scoring policy")

figures_manifest_df = pd.DataFrame(figures_manifest)
display(figures_manifest_df)


## 12. Reporte narrativo automático


In [ ]:
# =========================================================
# 12. Narrative monitoring report
# =========================================================

def safe_markdown_table(df, max_rows=30):
    """
    Minimal markdown table renderer that does not require the optional 'tabulate' package.
    """
    if df is None or df.empty:
        return ""
    df_str = df.head(max_rows).copy().astype(str)
    cols = list(df_str.columns)
    header = "| " + " | ".join(cols) + " |"
    separator = "| " + " | ".join(["---"] * len(cols)) + " |"
    rows = []
    for _, row in df_str.iterrows():
        rows.append("| " + " | ".join(row[c].replace("\n", " ") for c in cols) + " |")
    return "\n".join([header, separator] + rows)


top_alerts = (
    validity_df[validity_df["auth_recommendation_validity_status"].isin(["Red", "Yellow", "Watch", "Not_Evaluable"])]
    .sort_values(["auth_recommendation_validity_status", "baseline_revenue_sum_numeric"], ascending=[False, False])
    .head(30)
)

top_history_alerts = (
    input_history_drift_log[input_history_drift_log["drift_status"].isin(["Red", "Yellow"])]
    .sort_values(["drift_status", "drift_value"], ascending=[False, False])
    .head(15)
)

report_lines = []
report_lines.append("# Avance 4 - AUTH Recommendation Validity vs Current AUTH History REAL v4")
report_lines.append("")
report_lines.append(f"**Drift run ID:** {DRIFT_RUN_ID}")
report_lines.append(f"**Timestamp UTC:** {RUN_TIMESTAMP_UTC.isoformat()}")
report_lines.append(f"**Monitoring scope:** {MONITORING_SCOPE}")
report_lines.append(f"**Runs model:** {RUNS_MODEL}")
report_lines.append(f"**Baseline snapshot:** `{BASELINE_SNAPSHOT_PATH}`")
report_lines.append(f"**Current AUTH history snapshot:** `{CURRENT_AUTH_HISTORY_PATH}`")
report_lines.append(f"**Recommendation source column:** `{RECOMMENDATION_PRICE_COLUMN}`")
report_lines.append(f"**Alert profile:** `{ALERT_PROFILE}`")
report_lines.append(f"**Bin catalog scope:** `{BIN_CATALOG_SCOPE}`")
report_lines.append(f"**Bin catalog version:** `{BIN_CATALOG_VERSION}`")
report_lines.append("")

report_lines.append("## Interpretación del alcance")
report_lines.append("")
report_lines.append("Esta etapa utiliza únicamente datos AUTH. Por lo tanto, P20/P50/P85 representan historia de precios autorizados, no mercado completo. El objetivo es validar si la recomendación vigente sigue siendo coherente con la historia AUTH actual real observada.")
report_lines.append("")
report_lines.append("Los catálogos de bins fueron creados con todos los datos disponibles (AUTH + no AUTH). En este notebook no se recalculan esos catálogos; se monitorea la distribución de registros/snapshots AUTH dentro de los bins ya definidos/versionados.")
report_lines.append("")
report_lines.append("La versión v3 separa bandas válidas de bandas colapsadas. Si P20=P85, el caso se trata como `Single_Price_Regime` y no como alerta de extremo P20/P85.")
report_lines.append("")

report_lines.append("## Resumen global")
report_lines.append("")
for key in [
    "baseline_recommendation_rows", "current_auth_history_rows", "new_combo_count", "new_combo_rate",
    "current_history_coverage_rate", "data_quality_status", "recommendation_coverage_status",
    "catalog_bin_coverage_status", "auth_history_drift_status", "recommendation_validity_global_status",
    "price_drift_status", "run_readiness_status", "recommended_operational_action", "scoring_update_recommendation", "hb_svi_retraining_recommendation", "catalog_rebaseline_recommendation", "run_readiness_decision", "green_recommendation_rate", "watch_recommendation_rate",
    "yellow_recommendation_rate", "red_recommendation_rate", "not_evaluable_recommendation_rate",
    "red_revenue_share", "yellow_revenue_share", "watch_revenue_share",
]:
    report_lines.append(f"- **{key}:** {run_readiness_summary.get(key)}")
report_lines.append("")

report_lines.append("## Decisión operativa")
report_lines.append("")
report_lines.append("Esta versión separa `scoring/update` de `HB-SVI retraining`. Los nuevos combos son una señal de cobertura de recomendaciones, no una falla automática del modelo.")
report_lines.append("")
report_lines.append(safe_markdown_table(operational_decision_summary))
report_lines.append("")
report_lines.append("### Política sugerida para nuevos combos")
report_lines.append(safe_markdown_table(new_combo_operational_summary))
report_lines.append("")
report_lines.append(safe_markdown_table(new_combo_scoring_policy_summary))
report_lines.append("")

report_lines.append("## Resumen de banda y extremos")
report_lines.append("")
report_lines.append("### Estado de banda actual")
report_lines.append(safe_markdown_table(band_state_summary))
report_lines.append("")
report_lines.append("### Transición de extremos baseline vs current")
report_lines.append(safe_markdown_table(edge_transition_summary))
report_lines.append("")
report_lines.append("### Razones principales de vigencia")
report_lines.append(safe_markdown_table(validity_reason_summary.head(20)))
report_lines.append("")

report_lines.append("## Sensibilidad de alertas")
report_lines.append("")
report_lines.append("La sensibilidad se evalúa con tres perfiles. El perfil activo para esta corrida es el indicado arriba. Esto ayuda a decidir si Yellow debe ser vigilancia amplia o revisión accionable.")
report_lines.append(safe_markdown_table(sensitivity_profile_summary))
report_lines.append("")

report_lines.append("## Principales alertas de vigencia de recomendación")
report_lines.append("")
if top_alerts.empty:
    report_lines.append("No se detectaron recomendaciones en estado Watch, Yellow, Red o Not_Evaluable.")
else:
    cols = KEY_COLUMNS + [
        "baseline_recommended_price", "baseline_P20_PRICE", "baseline_P50_PRICE", "baseline_P85_PRICE",
        "current_P20_PRICE", "current_P50_PRICE", "current_P85_PRICE",
        "current_band_state", "edge_transition_type", "recommendation_position_baseline_auth_band",
        "recommendation_position_current_auth_band", "auth_recommendation_validity_status",
        "auth_recommendation_validity_reason", "recommended_action", "baseline_revenue_sum_numeric"
    ]
    cols = [c for c in cols if c in top_alerts.columns]
    report_lines.append(safe_markdown_table(top_alerts[cols], max_rows=30))
report_lines.append("")

report_lines.append("## Principales alertas de historia AUTH")
report_lines.append("")
if top_history_alerts.empty:
    report_lines.append("No se detectaron métricas de historia AUTH en estado Yellow o Red.")
else:
    cols = ["monitoring_stage", "variable_name", "drift_metric", "drift_value", "drift_status", "recommended_action"]
    report_lines.append(safe_markdown_table(top_history_alerts[cols]))
report_lines.append("")

report_lines.append("## Principales cambios en bins globales sobre AUTH")
report_lines.append("")
if bin_drift_summary.empty:
    report_lines.append("No se encontraron columnas de bins disponibles.")
else:
    report_lines.append(safe_markdown_table(bin_drift_summary.head(15)))
report_lines.append("")

report_lines.append("## Evidencia visual")
report_lines.append("")
report_lines.append(f"Las figuras fueron guardadas en `{OUTPUT_DIRS['figures']}` y registradas en `figures_manifest.csv`.")
report_lines.append("")

report_lines.append("## Limitaciones")
report_lines.append("")
report_lines.append("- Este notebook no reentrena ni corre el modelo; solo evalúa vigencia de recomendaciones existentes.")
report_lines.append("- `Watch` no equivale a error. Es una zona de vigilancia que evita sobrecargar Yellow con casos no accionables.")
report_lines.append("- Los casos `Collapsed_Band` o `Single_Price_Regime` no se interpretan como extremos P20/P85. Se evalúan por gap contra P50.")
report_lines.append("- Nuevos combos sin recomendación baseline se interpretan como cobertura de recomendación, no como data quality defect.")
report_lines.append("- El reentrenamiento HB-SVI debe evaluarse con model health/training diagnostics; una nueva corrida de scoring puede justificarse por cobertura aunque no haya drift fuerte de precios.")
report_lines.append("")

report_lines.append("## Siguiente paso recomendado")
report_lines.append("")
report_lines.append(run_readiness_decision)

report_text = "\n".join(report_lines)
print(report_text[:4000])
print("\nReport prepared for materialization under reports/auth_recommendation_validity_report.md")


## 13. Exportacion local de artefactos MLOps

**Platform-owned artifact publishing:** esta copia de transicion solo materializa artefactos locales siguiendo `MONITORING_ARTIFACTS`. La publicacion a Azure Blob, Azure ML tags, SQL o cualquier sink final pertenece a `pricing-mlops-platform`.

Cuando una seccion sea abstraida a componente, este notebook debe reemplazar la logica inline por una llamada al modulo correspondiente y mantener el mismo contrato local de salida.


In [ ]:
# =========================================================
# 13. Materialize monitoring artifacts
# =========================================================

# Export local de artefactos. La publicacion final a Azure/SQL/MLflow pertenece al repo
# pricing-mlops-platform; esta celda solo materializa el contrato local del pipeline.
materialized_outputs = materialize_monitoring_outputs(
    output_root=OUTPUT_ROOT,
    snapshots={
        "baseline_recommendation_snapshot.csv": baseline_recommendation_snapshot,
        "baseline_auth_history_profile.csv": baseline_auth_history_profile,
        "current_auth_history_snapshot_real.csv": current_auth_history,
    },
    logs={
        "auth_recommendation_validity_log.csv": validity_df,
        "new_combo_without_baseline_recommendation_log.csv": new_combo_log,
        "data_quality_log.csv": data_quality_log,
        "recommendation_coverage_log.csv": recommendation_coverage_log,
        "catalog_bin_coverage_log.csv": catalog_bin_coverage_log,
        "auth_history_drift_log.csv": input_history_drift_log,
    },
    summaries={
        "quality_status_summary.csv": quality_status_summary,
        "recommendation_coverage_summary.csv": recommendation_coverage_summary,
        "catalog_bin_coverage_summary.csv": catalog_bin_coverage_summary,
        "auth_recommendation_validity_summary.csv": validity_summary,
        "auth_recommendation_validity_revenue_summary.csv": validity_revenue_summary,
        "auth_recommendation_validity_reason_summary.csv": validity_reason_summary,
        "edge_transition_summary.csv": edge_transition_summary,
        "current_band_state_summary.csv": band_state_summary,
        "sensitivity_profile_summary.csv": sensitivity_profile_summary,
        "segment_validity_summary.csv": segment_validity_summary,
        "global_bin_drift_log.csv": global_bin_drift_log,
        "global_bin_drift_summary.csv": global_bin_drift_summary,
        "new_combo_operational_summary.csv": new_combo_operational_summary,
        "new_combo_scoring_policy_summary.csv": new_combo_scoring_policy_summary,
        "existing_recommendation_risk_summary.csv": existing_recommendation_risk_summary,
        "run_readiness_summary.csv": run_readiness_summary_df,
        "operational_decision_summary.csv": operational_decision_summary,
    },
    reports={
        "auth_recommendation_validity_report.md": report_text,
    },
)

print("Artifacts exported to:", materialized_outputs.output_root)
print("Materialized files:", len(materialized_outputs.relative_paths))


## 14. Artifact manifest


In [ ]:
# =========================================================
# 14. Artifact manifest
# =========================================================

# Manifest local compatible con platform_publish_outputs.py: paths relativos bajo OUTPUT_ROOT.
artifact_manifest_json_path = write_artifact_manifest(OUTPUT_ROOT, DRIFT_RUN_ID)
artifact_manifest_payload = json.loads(artifact_manifest_json_path.read_text(encoding="utf-8"))
artifact_manifest = pd.DataFrame(artifact_manifest_payload["artifacts"])

artifact_manifest_csv_path = OUTPUT_DIRS["manifest"] / "artifact_manifest.csv"
artifact_manifest.to_csv(artifact_manifest_csv_path, index=False)

display(artifact_manifest)
print("Artifact manifest exported:", artifact_manifest_json_path)
print("Artifact manifest table exported:", artifact_manifest_csv_path)


## 15. Validacion del contrato de artefactos

Este bloque usa `MONITORING_ARTIFACTS` para comprobar que los outputs requeridos por el pipeline fueron materializados localmente antes de que plataforma los publique.


In [ ]:

# =========================================================
# 15. Validate expected monitoring artifacts
# =========================================================

artifact_contract_validation = validate_expected_monitoring_artifacts(OUTPUT_ROOT, MONITORING_ARTIFACTS)
artifact_contract_validation_df = pd.DataFrame(artifact_contract_validation.records)
display(artifact_contract_validation_df)

if artifact_contract_validation.missing_required:
    missing = [record["relative_path"] for record in artifact_contract_validation.missing_required]
    raise RuntimeError(f"Missing required monitoring artifacts: {missing}")

print("Required monitoring artifacts complete:", artifact_contract_validation.is_complete)


## 16. Cierre del Avance 4

Este notebook deja implementada la compuerta principal de monitoreo pre-model para el alcance AUTH:

1. Valida calidad de datos.
2. Evalúa drift de historia AUTH.
3. Evalúa si la recomendación vigente sigue alineada con la historia AUTH actual.
4. Identifica casos y segmentos que requieren revisión.
5. Exporta artefactos listos para almacenamiento en Azure Storage/ADLS y registro resumido en Azure SQL.

La lógica no reemplaza al modelo de recomendación. Su función es decidir si la recomendación vigente puede mantenerse, si requiere revisión o si se justifica una nueva corrida/recalibración.

## Lectura final recomendada

La etapa queda cerrada como una compuerta AUTH-only pre-model. El resultado Red/Yellow/Green no debe entenderse como un juicio absoluto del modelo, sino como una señal de operación:

- **Green:** la recomendación vigente puede mantenerse con monitoreo normal.
- **Yellow:** la recomendación sigue siendo defendible, pero requiere revisión por cambio moderado en historia AUTH o cercanía a extremos.
- **Red:** la recomendación vigente requiere revisión antes de confiar en ella; puede justificar una nueva corrida, recalibración o revisión con PM/Pricing.

El siguiente trabajo ya no es seguir modificando la lógica del notebook, sino conectarla a la arquitectura MLOps: Storage/ADLS para evidencia completa, Azure SQL para metadatos y resúmenes, y dashboard para monitoreo operativo.


**Nota v4:** el monitoreo concluye con una acción operativa separada: mantener/vigilar, revisar casos accionables, correr scoring/update para nuevos combos, revisar catálogo o evaluar reentrenamiento HB-SVI. El reentrenamiento no se dispara solo por calendario ni por nuevos combos; requiere señales de precio/model health o soporte suficiente.
